# Explorer 02 - Genres and styles

**Purpose.** Explore how genre and style composition moved from 2016 to 2026 and choose the fragmentation measure.

**Feeds main report section:** 5.1 RQ1 - Genre and style shifts

**Modules used:** `vinyl.trends`, `vinyl.viz`

**Tables read:** `release_genres`, `release_styles`

This is a working notebook. It can be messy; anything that goes into the report is re-run from `main.ipynb` through the same functions in `vinyl/`.

In [ ]:
# Make the `vinyl` package importable from the explorers/ folder.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "explorers":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from pyspark.sql import functions as F
from vinyl import config, ingest

spark = ingest.get_spark(app_name="vinyl-02-styles")
from vinyl import trends, viz

In [ ]:
snapshot = config.MAIN_SNAPSHOT  # "2026"
release_genres = ingest.read_table(spark, snapshot, "release_genres")
release_styles = ingest.read_table(spark, snapshot, "release_styles")
release_artists = ingest.read_table(spark, snapshot, "release_artists")

## 1. Coverage: how many releases have a genre or style at all?

In [ ]:
releases = ingest.read_table(spark, snapshot, "releases")
total = releases.groupBy("year").agg(F.count("*").alias("n_all"))
tagged = release_styles.groupBy("year").agg(F.countDistinct("release_id").alias("n_with_style"))
total.join(tagged, "year", "left").withColumn("coverage", F.col("n_with_style") / F.col("n_all")).orderBy("year").show()

## 2. Genre and style shares by year

In [ ]:
genre_shares = trends.tag_shares_by_year(release_genres, tag_col="genre").toPandas()
viz.plot_lines(
    genre_shares, "year", "share", "genre",
    title="Share of each year's releases tagged with each genre (2026 is partial: Jan-Sep)",
    ylabel="share of releases", percent=True,
)

In [ ]:
style_shares = trends.tag_shares_by_year(release_styles, tag_col="style")
start_year, last_full, prev_year = 2016, config.LAST_FULL_YEAR, config.LAST_FULL_YEAR - 1
movers = {
    end: trends.top_movers(style_shares, start_year=start_year, end_year=end, min_releases=500).toPandas()
    for end in (last_full, prev_year)
}
for end, df in movers.items():
    print(f"Style share change {start_year} -> {end}: 15 largest gains, then 15 largest losses")
    display(df.head(15))
    display(df.tail(15))

# Does the cataloguing lag in the last full year change the ranking?
rank = {end: df.set_index("style")["change"].rank(ascending=False) for end, df in movers.items()}
both = rank[last_full].to_frame(f"rank_{last_full}").join(rank[prev_year].rename(f"rank_{prev_year}"), how="inner")
top = {end: set(df.head(15)["style"]) for end, df in movers.items()}
print(f"Spearman rank correlation of the change, {last_full} vs {prev_year} end year: "
      f"{both.corr(method='spearman').iloc[0, 1]:.3f}")
print(f"Styles in both top-15 gain lists: {len(top[last_full] & top[prev_year])} of 15")

### 2b. Robustness: drop the prolific artists

A few real artists self-release hundreds or thousands of releases in one year. In 2026, 35 artist-years reach `config.PROLIFIC_ARTIST_MIN_PER_YEAR` (300) releases, 0.4% of all releases, so totals barely move. Niche styles can: artist 6351861 alone is 22% of the 2019 "Harsh Noise Wall" releases and 27% of 2023.

The next cell recomputes the style shares and `top_movers` without the releases of those artist-years (`trends.drop_prolific_releases`). This is a robustness check, not the main definition. The main ranking stays the one above.

Reading rule:

- If a style's `change` shrinks a lot once the prolific artists are dropped, that style's trend is one artist's output, not a scene. Say so next to the style in the report.
- If the styles and the size of `change` stay about the same, the trend does not depend on them.
- A Spearman correlation close to 1 means the ranking as a whole is stable. Check the top and bottom 15 directly, because the correlation is dominated by the many styles in the middle.

In [ ]:
import pandas as pd

prolific = trends.prolific_artist_years(release_artists).cache()
styles_clean = trends.drop_prolific_releases(release_styles, release_artists, prolific)
shares_clean = trends.tag_shares_by_year(styles_clean, tag_col="style")
movers_clean = trends.top_movers(shares_clean, start_year=start_year, end_year=last_full, min_releases=500).toPandas()
movers_main = movers[last_full]

def side_by_side(main, clean):
    """Put the main and the prolific-dropped ranking next to each other."""
    left = main[["style", "change"]].reset_index(drop=True).add_suffix("_main")
    right = clean[["style", "change"]].reset_index(drop=True).add_suffix("_no_prolific")
    return left.join(right)

print(f"Style share change {start_year} -> {last_full}: main ranking (left) vs without prolific artists (right)")
print("15 largest gains")
display(side_by_side(movers_main.head(15), movers_clean.head(15)))
print("15 largest losses")
display(side_by_side(movers_main.tail(15), movers_clean.tail(15)))

# Change of the main top and bottom 15 once the prolific artists are dropped.
watch = pd.concat([movers_main.head(15), movers_main.tail(15)])[["style", "change"]]
watch = watch.merge(movers_clean[["style", "change"]], on="style", how="left", suffixes=("_main", "_no_prolific"))
watch["shrunk_by"] = watch["change_main"].abs() - watch["change_no_prolific"].abs()
print("Styles of the main top/bottom 15, largest shrinkage first")
display(watch.sort_values("shrunk_by", ascending=False).head(10))

both = movers_main.set_index("style")["change"].to_frame("main").join(movers_clean.set_index("style")["change"].rename("no_prolific"), how="inner")
print(f"Spearman correlation of the change, main vs without prolific artists: {both.corr(method='spearman').iloc[0, 1]:.3f} ({len(both)} styles in both)")

print(f"Prolific artist-years (>= {config.PROLIFIC_ARTIST_MIN_PER_YEAR} releases in one year)")
display(prolific.toPandas().sort_values("n_releases", ascending=False))

## 3. Fragmentation: Shannon entropy

In [ ]:
entropy = trends.shannon_entropy_by_year(release_styles).toPandas()
entropy

In [ ]:
viz.plot_lines(
    entropy, "year", "evenness",
    title="Style evenness (entropy / max entropy) by year (2026 is partial: Jan-Sep)",
    ylabel="evenness (0-1)",
)

## 4. Style co-occurrence networks

In [ ]:
cooc = trends.style_cooccurrence(release_styles, min_count=50).cache()
cooc.where(F.col("year") == config.LAST_FULL_YEAR).orderBy(F.desc("jaccard")).show(20, truncate=False)

In [ ]:
graph_stats = trends.cooccurrence_graph_stats(cooc)
graph_stats

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
viz.plot_lines(graph_stats, "year", "density", title="Style co-occurrence network density (2026 partial)", ax=axes[0])
viz.plot_lines(graph_stats, "year", "modularity", title="Louvain modularity of the network (2026 partial)", ax=axes[1])
fig.tight_layout()

Edges are kept at `min_count=50` releases per year, an absolute threshold. Years with fewer releases keep fewer edges, so compare `density` and `n_nodes` across years with similar release counts, and treat 2026 as partial.

## 5. Fragmentation: new styles or evener use of old ones?

Reading rule for `entropy_decomposition`:

- `entropy_base_tags` is the entropy over only the styles that already existed in the base year (2016), renormalized.
- If `entropy_base_tags` rises along with `entropy`, the change comes from evener use of old styles.
- If `entropy_base_tags` stays flat while `entropy` rises, the change comes from new styles appearing. `n_new_tags` and `new_tag_pair_share` show how many there are and how much of the tagging they carry.

In [ ]:
decomp = trends.entropy_decomposition(release_styles, tag_col="style", base_year=2016).toPandas()
decomp

In [ ]:
bits = decomp.melt(id_vars="year", value_vars=["entropy", "entropy_base_tags"], var_name="measure", value_name="bits")
viz.plot_lines(
    bits, "year", "bits", "measure",
    title="Style entropy: all styles vs styles that existed in 2016 (2026 partial)",
    ylabel="Shannon entropy (bits)",
)

## 6. Sensitivity to undercounted years

Recent years have fewer catalogued releases, and 2026 is partial. `entropy_sensitivity` redraws, for every year, random samples of the same size (the smallest full year in 2016-`config.LAST_FULL_YEAR`) and recomputes evenness. Evenness depends on sample size, so compare the shape of the sampled line (and its min-max band) with the full-data line, not their levels. If both change by the same amount between years, the trend is not an effect of sample size. If only the full-data line moves, it is. A year with no more releases than the sample size is used whole, so its sampled values equal the full-data value.

In [ ]:
sensitivity = trends.entropy_sensitivity(release_styles, releases, tag_col="style", n_draws=5, seed=42).toPandas()
sensitivity

In [ ]:
lines = sensitivity.melt(id_vars="year", value_vars=["evenness_full", "evenness_sub_mean"], var_name="series", value_name="evenness")
ax = viz.plot_lines(
    lines, "year", "evenness", "series",
    title="Style evenness: all releases vs equal-size samples (min-max band; 2026 partial)",
)
ax.fill_between(sensitivity["year"], sensitivity["evenness_sub_min"], sensitivity["evenness_sub_max"], alpha=0.25, color="grey")

## Decisions to carry into `main.ipynb`

Filled in from the full-data runs in `executed/`. Each line should end up as a sentence in the report's methodology or limitations.

- [x] Headline fragmentation number: evenness (entropy divided by the log of the number of styles used that year), because that number changes from year to year (700-729 styles), so raw entropy is not comparable; entropy is reported next to it (7.32 bits in 2016, 7.47 in 2025; evenness 0.772 to 0.785).
- [x] End year for trend comparisons: 2025 (`config.LAST_FULL_YEAR`), since 2026 is partial. Ending in 2024 instead gives a Spearman correlation of 0.926 for the style changes and 12 of the same 15 top gainers, so the lag in 2025 does not change the picture.
- [x] Styles worth plotting: no single-style chart; the report shows the 15 largest gains and losses as a table (Hardcore +1.1, Alt-Pop +1.0, Black Metal +0.7 percentage points; Experimental -1.4, Electro House -1.0, House -0.8).
- [x] `min_count` for co-occurrence edges: 50 releases per year, giving 2,048 edges in 2016 and 1,530 in 2025 over 294-321 styles in the full years. The threshold is absolute, so years with fewer releases keep fewer edges.